In [ ]:
import subprocess
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'models' / 'src'))

from common import contract, data, metrics, paths, plots, verify

import rf

RAIZ = paths.ROOT               # os notebooks moram em utils/
PROCESSADO = paths.SPLITS

LIB = paths.LIB_RF
SRC = paths.LIB_RF_SRC

SCRATCH = paths.SCRATCH

SEMENTE = contract.SEED
ALVO = contract.TARGET
FEATURES = contract.RF_FEATURES
GRANULARIDADES = contract.GRANULARIDADES          # ("5min", "10min", "1h", "1D")
ROTULO = contract.GRANULARIDADE_LABEL

# O treino so' gera os joblibs em dist/. A biblioteca C do firmware sai pelo
# SCRIPT SEPARADO utils/gerar_lib_rf.py, que carrega o joblib da granularidade
# embarcada (o firmware leva uma lib por vez).

PCA_MIN, PCA_MAX = data.physical_target_limits()

np.random.seed(SEMENTE)

print("features:", ", ".join(FEATURES))
print("splits  : train (2021..2024H1) | calibration (2024H2) | test (2025)")
print(f"granularidades do treino: {', '.join(ROTULO[f] for f in GRANULARIDADES)}")
print("lib C: gerada pelo script utils/gerar_lib_rf.py a partir dos joblibs")
print("g++ host:", subprocess.run([verify.GXX, "--version"], capture_output=True,
                                 text=True).stdout.splitlines()[0])
print("xtensa  :", "ok" if verify.XTENSA_GXX.exists() else "AUSENTE")
print("entrega :", LIB.relative_to(RAIZ))


In [ ]:
# Um dicionario por granularidade, como no treino_mlp: splits, z-score com
# constantes SO do treino e o BRUTO do teste (para o conferidor da lib).
dados = {}
for freq in GRANULARIDADES:
    treino = data.load_split("train", freq=freq)
    calibracao = data.load_split("calibration", freq=freq)
    teste = data.load_split("test", freq=freq)

    # Sanidade: o csv precisa estar em GRANDEZA FISICA. Se alguem voltar a
    # normalizar no prep, estes limites quebram AQUI, e nao em campo, com o
    # firmware normalizando duas vezes e a previsao colapsando num valor fixo.
    for nome, d in (("train", treino), ("calibration", calibracao), ("test", teste)):
        data.check_physical_scale(f"{freq}/{nome}", d)

    MEDIA, DESVIO = data.fit_normalization(treino)
    dados[freq] = dict(
        treino=treino, calibracao=calibracao, teste=teste,
        media=MEDIA, desvio=DESVIO,
        X_tre=data.z_score(treino, MEDIA, DESVIO).to_numpy(np.float32),
        y_tre=treino[ALVO].to_numpy(np.float32),
        X_cal=data.z_score(calibracao, MEDIA, DESVIO).to_numpy(np.float32),
        y_cal=calibracao[ALVO].to_numpy(np.float32),
        X_tes=data.z_score(teste, MEDIA, DESVIO).to_numpy(np.float32),
        y_tes=teste[ALVO].to_numpy(np.float32),
        bruto_tes=teste[list(FEATURES)].to_numpy(np.float32),
    )
    print(f"{ROTULO[freq]:>6} | treino {len(dados[freq]['y_tre']):6} "
          f"| calibracao {len(dados[freq]['y_cal']):5} "
          f"| teste {len(dados[freq]['y_tes']):5} "
          f"| PCA {dados[freq]['y_tre'].min():.3f}..{dados[freq]['y_tre'].max():.2f} kW")

print("OK: os csv estao em grandeza fisica (brutos); z-score ajustado so no treino")


In [ ]:
from common.metrics import evaluate

resultados = {}      # resultados[freq] = lista de metricas por modelo


In [ ]:
from sklearn.linear_model import LinearRegression

# Linha de base de referencia: regressao linear multipla nas mesmas features.
# Treino imediato e um acerto que qualquer modelo nao-linear precisa superar
# para justificar a complexidade.
for freq in GRANULARIDADES:
    d = dados[freq]
    lin = LinearRegression().fit(d["X_tre"], d["y_tre"])
    pred = lin.predict(d["X_tes"]).ravel()
    print(freq, end='\t')
    resultados[freq] = [evaluate(f"regressao linear", d["y_tes"], pred)]

print("baseline ajustada nas quatro granularidades")


In [ ]:
import joblib
from sklearn.ensemble import RandomForestRegressor

paths.DIST.mkdir(parents=True, exist_ok=True)
varreduras, configuracoes, importancias = {}, {}, {}

# Varredura + treino final, por granularidade, numa execucao so. O vencedor
# sai do R2 JA CALIBRADO (offset do proprio candidato) na calibracao; as
# colunas de teste sao report-only. O modelo final e' salvo em dist/ ANTES de
# qualquer compilacao: a cruzada e a etapa da lib carregam o joblib.
for freq in GRANULARIDADES:
    d = dados[freq]
    print(f"===== {ROTULO[freq]} =====")
    print(f"{'config':>22} {'nos':>8} {'prof':>5} {'KB (est.)':>10} {'MAE':>8} {'R2':>8} "
          f"{'offset':>8} {'MAE cal':>8} {'R2 cal':>8} {'MAE teste':>10} {'R2 teste':>9} "
          f"{'s':>5} {'s_inf (us)':>10}")
    print("-" * 133)

    varredura = rf.sweep(d["X_tre"], d["y_tre"], d["X_cal"], d["y_cal"],
                         d["X_tes"], d["y_tes"])
    varreduras[freq] = varredura.assign(freq=freq, granularidade=ROTULO[freq])
    for r_ in varredura.itertuples():
        prof = "-" if pd.isna(r_.prof_max) else f"{int(r_.prof_max)}"
        print(f"{f'{r_.arvores}arv folha={r_.folha} d={prof}':>22} {r_.nos:8} {r_.prof:5} "
              f"{r_.kb:10.1f} {r_.mae:8.4f} {r_.r2:8.4f} {r_.offset:+8.4f} "
              f"{r_.mae_cal:8.4f} {r_.r2_cal:8.4f} {r_.mae_tes:10.4f} {r_.r2_tes:9.4f} "
              f"{r_.s:5.1f} {r_.s_inf * 1e6:10.1f}")

    top2 = varredura.nlargest(2, "r2_cal")
    # Desempate pelo MENOR tempo de inferencia por amostra (s_inf, medido na
    # calibracao); se a medicao faltar, cai para a menor quantidade de nos.
    if top2["s_inf"].notna().all():
        melhor = top2.loc[top2.s_inf.idxmin()]
    else:
        melhor = top2.loc[top2.nos.idxmin()]
    print(f"melhor na calibracao (entre os 2 maiores R2 calibrado, menor tempo de "
          f"inferencia): {int(melhor.arvores)} arvores, folha={int(melhor.folha)} "
          f"(R2 cal={melhor.r2_cal:.4f}, MAE cal={melhor.mae_cal:.4f}, "
          f"s_inf={melhor.s_inf * 1e6:.1f} us/amostra) com {melhor.kb:.0f} KB estimados")

    t0 = time.perf_counter()
    modelo = RandomForestRegressor(
        n_jobs=-1,
        n_estimators=int(melhor.arvores),
        min_samples_leaf=int(melhor.folha),
        # O vencedor da varredura pode ter teto de profundidade (max_depth): o
        # modelo final tem que ser O MESMO que a varredura avaliou.
        max_depth=None if pd.isna(melhor.prof_max) else int(melhor.prof_max),
        random_state=contract.SEED,
    ).fit(d["X_tre"], d["y_tre"])
    t_treino = time.perf_counter() - t0
    print(f"treino: {t_treino:.1f} s")

    # Calibracao do vies: offset = media(real - previsto) na calibracao (2024H2).
    # O modelo foi treinado ate' 2024H1; 2024H2 ja' mostra a queda de eficiencia.
    pred_cal = modelo.predict(d["X_cal"])
    offset = float(np.mean(d["y_cal"] - pred_cal))

    pred_rf = modelo.predict(d["X_tes"]) + offset
    d["pred_tes"] = pred_rf
    resultados[freq].append(evaluate("random forest", d["y_tes"], pred_rf))
    print(f"offset de calibracao (2024H2): {offset:+.4f} kW")

    n_nos = rf.count_nodes(modelo)
    print(f"{n_nos} nos em {modelo.n_estimators} arvores "
          f"({n_nos // modelo.n_estimators} por arvore, profundidade maxima "
          f"{rf.tree_depth(modelo)})")

    importancia = pd.Series(modelo.feature_importances_, index=FEATURES).sort_values()
    importancias[freq] = importancia
    print("importancia:", importancia.round(4).to_dict())

    configuracoes[freq] = {
        "granularidade": ROTULO[freq],
        "arvores": int(modelo.n_estimators),
        "folha_min": int(modelo.min_samples_leaf),
        "profundidade_max": modelo.max_depth,
        "nos": n_nos,
        "profundidade": rf.tree_depth(modelo),
        "offset_kw": offset,
        "kb_estimado": float(melhor.kb),
        "s_inf_us": float(melhor.s_inf * 1e6) if pd.notna(melhor.s_inf) else float("nan"),
        "treino_s": t_treino,
    }

    # Salva ANTES de compilar: o .joblib guarda o estimador e as constantes
    # necessarias para reproduzir a previsao em Python (e para a cruzada).
    caminho_modelo = paths.DIST / f"random_forest_{freq}.joblib"
    joblib.dump({
        "modelo": modelo,
        "freq": freq,
        "features": list(FEATURES),
        "media": d["media"].to_dict(),
        "desvio": d["desvio"].to_dict(),
        "offset_calibracao_kw": offset,
        "pca_min_kw": float(PCA_MIN),
        "pca_max_kw": float(PCA_MAX),
        "metricas": resultados[freq][1],
    }, caminho_modelo)
    print(f"modelo salvo: {caminho_modelo.relative_to(RAIZ)} "
          f"({caminho_modelo.stat().st_size / 1024:.1f} KB)")

In [ ]:
# Hexbins de real x predito por granularidade (mesmo formato da figura do
# MLP na monografia).
fig, eixos = plt.subplots(2, 2, figsize=(13, 10))
for eixo, freq in zip(eixos.ravel(), GRANULARIDADES):
    d = dados[freq]
    r = resultados[freq][1]
    plots.hexbin_real_vs_pred(eixo, d["y_tes"], d["pred_tes"],
                              f"RF no teste, {ROTULO[freq]} (kW)")
    texto = chr(10).join([
        f"MAE {r['MAE (kW)']:.2f} kW",
        f"RMSE {r['RMSE (kW)']:.2f} kW",
        f"R2 {r['R2']:.3f}",
        f"vies {r['vies (kW)']:+.2f} kW",
    ])
    eixo.text(0.03, 0.97, texto, transform=eixo.transAxes, va="top",
              color=plots.TINTA_2, fontsize=9, bbox=plots.HALO)
fig.tight_layout()
plots.salvar_figura(fig, "rf_avaliacao_granularidades.png")
plt.show()

In [ ]:
import joblib

from common import results

# Analise cruzada de granularidades: cada modelo treinado avalia no teste das
# QUATRO granularidades, carregado do joblib salvo (o mesmo artefato que a
# etapa da lib usa). O teste de B e' reescalado com o z-score do treino de A
# (fora da faixa usual quando B e' mais fino que A: o modelo extrapola, e e'
# exatamente isso que a analise mede). A comparacao e' pelo R2. A diagonal e'
# o proprio modelo; abaixo dela, modelo grosso avaliado em dado fino.
modelos = {freq: joblib.load(paths.DIST / f"random_forest_{freq}.joblib")
           for freq in GRANULARIDADES}

cruzada = []
for freq_a in GRANULARIDADES:
    pkg_a = modelos[freq_a]
    media_a = pd.Series(pkg_a["media"])
    desvio_a = pd.Series(pkg_a["desvio"])
    linha = {"treinado": ROTULO[freq_a]}
    for freq_b in GRANULARIDADES:
        dB = dados[freq_b]
        X_b = data.z_score(dB["teste"], media_a, desvio_a).to_numpy(np.float32)
        pred = pkg_a["modelo"].predict(X_b) + pkg_a["offset_calibracao_kw"]
        r = evaluate(f"A={freq_a} em B={freq_b}", dB["y_tes"], pred)
        linha[ROTULO[freq_b]] = r["R2"]
    cruzada.append(linha)

tabela_cruzada = pd.DataFrame(cruzada)
colunas = ["treinado", *[ROTULO[f] for f in GRANULARIDADES]]
print("R2 por par treinado x avaliado:")
print(tabela_cruzada[colunas].round(4).to_string(index=False))
results.salvar_tabela(tabela_cruzada[colunas], "cruzada_rf", "treino_rf")

fig, eixo = plt.subplots(figsize=(7, 5.5))
m = tabela_cruzada.set_index("treinado")[[ROTULO[f] for f in GRANULARIDADES]]
eixo.imshow(m.to_numpy(), cmap="viridis_r")
eixo.set_xticks(range(len(GRANULARIDADES)), [ROTULO[f] for f in GRANULARIDADES])
eixo.set_yticks(range(len(GRANULARIDADES)), m.index)
meio = (m.to_numpy().min() + m.to_numpy().max()) / 2
for i in range(len(m)):
    for j in range(len(m.columns)):
        eixo.text(j, i, f"{m.iloc[i, j]:.3f}", ha="center", va="center",
                  color="white" if m.iloc[i, j] > meio else "black")
fig.tight_layout()
plots.salvar_figura(fig, "rf_cruzada.png")
plots.enfeitar(eixo, "R2: treinado x avaliado",
               "avaliado em", "treinado em")
plt.show()

In [ ]:
from common import results

print(f"{'modelo':>26} {'granularidade':>14} {'MAE':>9} {'RMSE':>9} {'R2':>9} {'MAE c/ clip':>12} {'vies':>9}")
print("-" * 96)
for freq in GRANULARIDADES:
    for r_ in resultados[freq]:
        print(f"{r_['modelo']:>26} {ROTULO[freq]:>14} {r_['MAE (kW)']:9.4f} "
              f"{r_['RMSE (kW)']:9.4f} {r_['R2']:9.4f} "
              f"{r_['MAE com clip (kW)']:12.4f} {r_['vies (kW)']:+9.4f}")

# Resultados em results/treino_rf/: metricas, varredura, configuracoes e
# importancia das features. Os modelos/artefatos continuam em dist/.
tabela_resultados = pd.DataFrame([
    {"freq": freq, "granularidade": ROTULO[freq], **r_}
    for freq in GRANULARIDADES for r_ in resultados[freq]
])
results.salvar_tabela(tabela_resultados, "metricas_rf", "treino_rf")
results.salvar_tabela(pd.concat(varreduras.values(), ignore_index=True),
                      "varredura_rf", "treino_rf")
results.salvar_tabela(
    pd.DataFrame(configuracoes).T.rename_axis("freq").reset_index(),
    "configs_rf", "treino_rf")
results.salvar_tabela(
    pd.concat([s.rename("importancia").rename_axis("feature").reset_index()
               .assign(freq=freq) for freq, s in importancias.items()],
              ignore_index=True),
    "importancia_rf", "treino_rf")

### Etapa da lib (script separado)

A biblioteca C do firmware nao sai mais deste notebook: o script
`utils/gerar_lib_rf.py` carrega o joblib salvo e gera a lib do PlatformIO.

```
python utils/gerar_lib_rf.py 1h           # gera a lib da granularidade 1h
python utils/gerar_lib_rf.py 1h --medir   # gera e mede flash/RAM (xtensa)
```

O firmware embarca uma lib por vez; escolha a `freq` do ensaio.
